# Step 3–4: CLIP ViT-L/14 embeddings + folds (Colab, **GPU runtime: T4**, ~1.5 h)
**What this notebook does**
1. Clones this repo and unpacks the audit's pinned UTKFace archive (sha256-checked).
2. Smoke run: embeds 256 UTKFace images to prove the pipeline (QuickGELU asserted).
3. Embeds all 23,705 UTKFace images with frozen OpenAI CLIP ViT-L/14 (~20 min).
4. **Cross-check:** zero-shot margins from our embeddings must reproduce the audit's Phase 1 CLIP L/14 scores (max diff < 1e-3, identical decisions). If this fails, stop.
5. Builds the 5 frozen folds (seed 0).
6. Embeds FairFace's train split (~87k images, ~1 h) for the extra-data variant.
7. Writes `step3_outputs.zip` (aggregate JSON only) to Drive.

**Resumable:** finished steps are skipped. If Colab disconnects, reconnect (T4) and Run all.
**Ethics:** never display images. Embeddings, ids and folds stay on Drive (`MyDrive/robust_child_filtering_model/`), never in git.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, json, subprocess, zipfile, hashlib
os.environ['CRF_ENV'] = 'colab'
REPO = pathlib.Path('/content/robust_child_filtering_model')
URL = 'github.com/isaacmutuma/robust_child_filtering_model.git'
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/robust_child_filtering_model')
os.chdir('/content')
!rm -rf {REPO}
# Private repo: add a Colab secret GITHUB_TOKEN (key icon, left bar; fine-grained token,
# read-only Contents access to this one repo).
if subprocess.run(['git', 'clone', '-q', f'https://{URL}', str(REPO)]).returncode != 0:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    subprocess.run(['git', 'clone', '-q', f'https://x-access-token:{tok}@{URL}', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', f'https://{URL}'], check=True)
!git -C {REPO} log --oneline -1
os.chdir(REPO)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU: switch runtime to T4'
!pip -q install open_clip_torch==3.3.0 huggingface_hub
import open_clip; print('open_clip', open_clip.__version__)

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

def done(stem):
    """True if Drive holds a clean manifest for `stem` and its npz still matches it."""
    m = DRIVE_ROOT / 'results' / 'embeddings' / f'{stem}.json'
    if not m.exists(): return False
    d = json.loads(m.read_text())
    npz = DRIVE_ROOT / d['npz']['drive_path']
    return d['git']['dirty'] is False and npz.exists() and sha256(npz) == d['npz']['sha256']

def save_manifest(rel):
    """Copy a results JSON to Drive after checking it came from clean, committed code."""
    src = REPO / 'results' / rel
    d = json.loads(src.read_text())
    assert d['git']['dirty'] is False, 'code was modified in this runtime'
    dst = DRIVE_ROOT / 'results' / rel; dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text(src.read_text()); print('saved', rel, 'commit', d['git']['commit'][:7])
    return d

## 1. Unpack UTKFace + smoke run (256 images)

In [ ]:
!python scripts/unpack_utkface.py
!python scripts/extract_embeddings.py --dataset utkface --limit 256
d = json.loads((REPO / 'results/embeddings/utkface__clip_vitl14__smoke.json').read_text())
print({k: d[k] for k in ['n_embedded', 'n_failed', 'dim', 'norm', 'seconds']},
      'QuickGELU layers:', d['weights']['n_quick_gelu_layers'])
assert d['n_failed'] == 0 and d['dim'] == 768 and d['weights']['n_quick_gelu_layers'] == 36

## 2. Full UTKFace embeddings (~20 min)

In [ ]:
STEM = 'utkface__clip_vitl14'
if done(STEM):
    print('UTKFace embeddings already on Drive; skipping')
else:
    !python scripts/extract_embeddings.py --dataset utkface
    d = save_manifest(f'embeddings/{STEM}.json')
    assert d['n_embedded'] == 23705 and d['n_failed'] == 0, (d['n_embedded'], d['n_failed'])
    print('by group', d['n_by_group'], '\nteens 10-17', d['n_teens_10_17_by_group'])

## 3. Cross-check against the audit's Phase 1 CLIP L/14 run (must PASS)

In [ ]:
!python scripts/check_vs_phase1.py
d = save_manifest('embeddings/check_vs_phase1.json')
assert d['pass'], 'STOP: embeddings do not match Phase 1; tell Claude before going on'

## 4. Folds (frozen once made)

In [ ]:
!python scripts/make_folds.py
d = save_manifest('folds/utkface_folds_seed0.json')
print('fold sizes', d['fold_sizes'], '| merged keys', d['merged_rare_keys'])

## 5. FairFace train embeddings (~1 h; for the extra-data variant only)

In [ ]:
STEM = 'fairface_train__clip_vitl14'
if done(STEM):
    print('FairFace train embeddings already on Drive; skipping')
else:
    !python scripts/extract_embeddings.py --dataset fairface_train
    d = save_manifest(f'embeddings/{STEM}.json')
    print('n', d['n_embedded'], 'failed', d['n_failed'], d['fail_reasons'], '\nby group', d['n_by_group'])

## 6. Aggregate outputs for the repo (JSON only)

In [ ]:
out = DRIVE_ROOT / 'step3_outputs.zip'
files = sorted((DRIVE_ROOT / 'results').rglob('*.json'))
with zipfile.ZipFile(out, 'w') as z:
    for f in files:
        z.write(f, f.relative_to(DRIVE_ROOT))
with zipfile.ZipFile(out) as z:
    names = z.namelist()
assert all(n.endswith('.json') for n in names)
print(out, out.stat().st_size, 'bytes, sha256', sha256(out)); print('\n'.join(names))